# Multi-Objective Optimization for Software Project Scheduling Problem (SPSP)
## Real-World Instance Notebook (Complete 200 Tasks Evaluation & 3 Strategic Decision Schedules)

**Author:** Research Intern  
**Framework:** `pymoo` & `networkx`  
**Dataset Storage:** **100% Manually Embedded** in Python Code  
**Dataset Size Evaluated:** Full 200 Tasks, 30 Developers, 6 Teams, 220 Dependencies (196 Acyclic + 24 Cyclic)  
**Skill Constraint Mechanism (Option A):** Skill-Aware Decoder mapping $x_i \to$ qualified developers  
**Visual Gantt Charts:** Explicit **Task ID labels (`t0` to `t199`)** labeled on execution bars  
**Decision-Making Strategies Outputted:**
1. **Strategy A (Fastest Makespan):** $\min(f_1)$ — Minimizes total completion duration.
2. **Strategy B (Fairest Workload Allocation):** $\min(f_2)$ — Minimizes workload standard deviation.
3. **Strategy C (Lowest Coordination Breakdown Risk):** $\min(f_3)$ — Minimizes FMEA coordination breakdown risk.  

---

### 1. Hardcoded Real-World Instance Dataset & Data Models
This cell contains the complete real-world SPSP problem instance manually embedded into Python data structures:
- **6 Teams:** `team_0` to `team_5`.
- **30 Developers:** Skill profiles, team assignments, and experience levels (`A`, `I`, `B`).
- **200 Tasks:** Required skills (`UI/UX`, `Backend`, `DB`, `API`, `Integration`), story points (1 to 13), and coupling degrees.
- **220 Dependencies:** Precedence constraints (196 Acyclic + 24 Cyclic dependencies).

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import time
from dataclasses import dataclass
from typing import List, Dict

random.seed(42)
np.random.seed(42)

@dataclass
class Task:
    task_id: int
    name: str
    req_skill: str
    story_points: int
    coupling_degree: float

@dataclass
class Developer:
    dev_id: int
    name: str
    skills: List[str]
    exp_level: str
    team_id: int

@dataclass
class Team:
    team_id: int
    name: str

@dataclass
class Dependency:
    pred_id: int
    succ_id: int
    dep_type: str
    coupling_degree: float

# MANUALLY EMBEDDED REAL-WORLD DATASET
RAW_TEAMS = [{'team_id': 0, 'name': 'team_0'}, {'team_id': 1, 'name': 'team_1'}, {'team_id': 2, 'name': 'team_2'}, {'team_id': 3, 'name': 'team_3'}, {'team_id': 4, 'name': 'team_4'}, {'team_id': 5, 'name': 'team_5'}]

RAW_DEVELOPERS = [{'dev_id': 0, 'name': 'dev_0', 'team_id': 5, 'skills': ['DB'], 'exp_level': 'A'}, {'dev_id': 1, 'name': 'dev_1', 'team_id': 3, 'skills': ['DB', 'UI/UX'], 'exp_level': 'A'}, {'dev_id': 2, 'name': 'dev_2', 'team_id': 4, 'skills': ['Backend'], 'exp_level': 'I'}, {'dev_id': 3, 'name': 'dev_3', 'team_id': 3, 'skills': ['Backend'], 'exp_level': 'B'}, {'dev_id': 4, 'name': 'dev_4', 'team_id': 1, 'skills': ['API'], 'exp_level': 'I'}, {'dev_id': 5, 'name': 'dev_5', 'team_id': 1, 'skills': ['Backend', 'Integration'], 'exp_level': 'I'}, {'dev_id': 6, 'name': 'dev_6', 'team_id': 5, 'skills': ['Backend', 'API', 'Integration'], 'exp_level': 'I'}, {'dev_id': 7, 'name': 'dev_7', 'team_id': 4, 'skills': ['API', 'Backend'], 'exp_level': 'I'}, {'dev_id': 8, 'name': 'dev_8', 'team_id': 2, 'skills': ['Integration'], 'exp_level': 'I'}, {'dev_id': 9, 'name': 'dev_9', 'team_id': 0, 'skills': ['DB', 'Integration', 'UI/UX'], 'exp_level': 'B'}, {'dev_id': 10, 'name': 'dev_10', 'team_id': 0, 'skills': ['API', 'UI/UX'], 'exp_level': 'B'}, {'dev_id': 11, 'name': 'dev_11', 'team_id': 3, 'skills': ['Backend', 'API'], 'exp_level': 'B'}, {'dev_id': 12, 'name': 'dev_12', 'team_id': 1, 'skills': ['Integration', 'Backend'], 'exp_level': 'I'}, {'dev_id': 13, 'name': 'dev_13', 'team_id': 2, 'skills': ['Backend'], 'exp_level': 'A'}, {'dev_id': 14, 'name': 'dev_14', 'team_id': 2, 'skills': ['API', 'DB'], 'exp_level': 'A'}, {'dev_id': 15, 'name': 'dev_15', 'team_id': 4, 'skills': ['DB', 'Backend'], 'exp_level': 'I'}, {'dev_id': 16, 'name': 'dev_16', 'team_id': 0, 'skills': ['Integration'], 'exp_level': 'I'}, {'dev_id': 17, 'name': 'dev_17', 'team_id': 0, 'skills': ['Backend'], 'exp_level': 'A'}, {'dev_id': 18, 'name': 'dev_18', 'team_id': 5, 'skills': ['Backend'], 'exp_level': 'B'}, {'dev_id': 19, 'name': 'dev_19', 'team_id': 0, 'skills': ['UI/UX', 'Integration'], 'exp_level': 'A'}, {'dev_id': 20, 'name': 'dev_20', 'team_id': 4, 'skills': ['API', 'DB', 'UI/UX'], 'exp_level': 'A'}, {'dev_id': 21, 'name': 'dev_21', 'team_id': 5, 'skills': ['UI/UX'], 'exp_level': 'I'}, {'dev_id': 22, 'name': 'dev_22', 'team_id': 3, 'skills': ['Integration'], 'exp_level': 'I'}, {'dev_id': 23, 'name': 'dev_23', 'team_id': 5, 'skills': ['Integration'], 'exp_level': 'I'}, {'dev_id': 24, 'name': 'dev_24', 'team_id': 4, 'skills': ['Backend'], 'exp_level': 'B'}, {'dev_id': 25, 'name': 'dev_25', 'team_id': 1, 'skills': ['Backend'], 'exp_level': 'I'}, {'dev_id': 26, 'name': 'dev_26', 'team_id': 2, 'skills': ['UI/UX'], 'exp_level': 'I'}, {'dev_id': 27, 'name': 'dev_27', 'team_id': 2, 'skills': ['Integration'], 'exp_level': 'B'}, {'dev_id': 28, 'name': 'dev_28', 'team_id': 3, 'skills': ['Integration'], 'exp_level': 'B'}, {'dev_id': 29, 'name': 'dev_29', 'team_id': 1, 'skills': ['Integration'], 'exp_level': 'I'}]

RAW_TASKS = [{'task_id': 0, 'name': 'task_0', 'story_points': 2, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 1, 'name': 'task_1', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.5}, {'task_id': 2, 'name': 'task_2', 'story_points': 3, 'req_skill': 'DB', 'coupling_degree': 0.648533}, {'task_id': 3, 'name': 'task_3', 'story_points': 2, 'req_skill': 'UI/UX', 'coupling_degree': 0.595842}, {'task_id': 4, 'name': 'task_4', 'story_points': 1, 'req_skill': 'Backend', 'coupling_degree': 0.576794}, {'task_id': 5, 'name': 'task_5', 'story_points': 2, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 6, 'name': 'task_6', 'story_points': 1, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 7, 'name': 'task_7', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.672417}, {'task_id': 8, 'name': 'task_8', 'story_points': 1, 'req_skill': 'API', 'coupling_degree': 0.561801}, {'task_id': 9, 'name': 'task_9', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.60147}, {'task_id': 10, 'name': 'task_10', 'story_points': 5, 'req_skill': 'API', 'coupling_degree': 0.8262}, {'task_id': 11, 'name': 'task_11', 'story_points': 3, 'req_skill': 'Integration', 'coupling_degree': 0.597342}, {'task_id': 12, 'name': 'task_12', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.747377}, {'task_id': 13, 'name': 'task_13', 'story_points': 13, 'req_skill': 'Integration', 'coupling_degree': 0.85433}, {'task_id': 14, 'name': 'task_14', 'story_points': 5, 'req_skill': 'Integration', 'coupling_degree': 0.62332}, {'task_id': 15, 'name': 'task_15', 'story_points': 8, 'req_skill': 'API', 'coupling_degree': 0.879063}, {'task_id': 16, 'name': 'task_16', 'story_points': 3, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 17, 'name': 'task_17', 'story_points': 1, 'req_skill': 'Integration', 'coupling_degree': 0.564216}, {'task_id': 18, 'name': 'task_18', 'story_points': 1, 'req_skill': 'Integration', 'coupling_degree': 0.558044}, {'task_id': 19, 'name': 'task_19', 'story_points': 13, 'req_skill': 'Backend', 'coupling_degree': 0.680119}, {'task_id': 20, 'name': 'task_20', 'story_points': 1, 'req_skill': 'Integration', 'coupling_degree': 0.881586}, {'task_id': 21, 'name': 'task_21', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.623294}, {'task_id': 22, 'name': 'task_22', 'story_points': 2, 'req_skill': 'DB', 'coupling_degree': 0.777235}, {'task_id': 23, 'name': 'task_23', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.895222}, {'task_id': 24, 'name': 'task_24', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.663611}, {'task_id': 25, 'name': 'task_25', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.573302}, {'task_id': 26, 'name': 'task_26', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.5}, {'task_id': 27, 'name': 'task_27', 'story_points': 1, 'req_skill': 'Backend', 'coupling_degree': 0.708127}, {'task_id': 28, 'name': 'task_28', 'story_points': 1, 'req_skill': 'DB', 'coupling_degree': 0.694075}, {'task_id': 29, 'name': 'task_29', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.642592}, {'task_id': 30, 'name': 'task_30', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.731283}, {'task_id': 31, 'name': 'task_31', 'story_points': 8, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 32, 'name': 'task_32', 'story_points': 8, 'req_skill': 'DB', 'coupling_degree': 0.762248}, {'task_id': 33, 'name': 'task_33', 'story_points': 2, 'req_skill': 'Backend', 'coupling_degree': 0.574439}, {'task_id': 34, 'name': 'task_34', 'story_points': 2, 'req_skill': 'API', 'coupling_degree': 0.84763}, {'task_id': 35, 'name': 'task_35', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.729319}, {'task_id': 36, 'name': 'task_36', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.745464}, {'task_id': 37, 'name': 'task_37', 'story_points': 3, 'req_skill': 'Integration', 'coupling_degree': 0.5}, {'task_id': 38, 'name': 'task_38', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.619678}, {'task_id': 39, 'name': 'task_39', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.774211}, {'task_id': 40, 'name': 'task_40', 'story_points': 2, 'req_skill': 'DB', 'coupling_degree': 0.337036}, {'task_id': 41, 'name': 'task_41', 'story_points': 13, 'req_skill': 'Integration', 'coupling_degree': 0.82811}, {'task_id': 42, 'name': 'task_42', 'story_points': 3, 'req_skill': 'DB', 'coupling_degree': 0.754303}, {'task_id': 43, 'name': 'task_43', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.762024}, {'task_id': 44, 'name': 'task_44', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.522283}, {'task_id': 45, 'name': 'task_45', 'story_points': 5, 'req_skill': 'Backend', 'coupling_degree': 0.813437}, {'task_id': 46, 'name': 'task_46', 'story_points': 2, 'req_skill': 'API', 'coupling_degree': 0.812555}, {'task_id': 47, 'name': 'task_47', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 48, 'name': 'task_48', 'story_points': 13, 'req_skill': 'Integration', 'coupling_degree': 0.628752}, {'task_id': 49, 'name': 'task_49', 'story_points': 1, 'req_skill': 'API', 'coupling_degree': 0.851303}, {'task_id': 50, 'name': 'task_50', 'story_points': 1, 'req_skill': 'Integration', 'coupling_degree': 0.808911}, {'task_id': 51, 'name': 'task_51', 'story_points': 8, 'req_skill': 'API', 'coupling_degree': 0.515713}, {'task_id': 52, 'name': 'task_52', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.74411}, {'task_id': 53, 'name': 'task_53', 'story_points': 1, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 54, 'name': 'task_54', 'story_points': 8, 'req_skill': 'API', 'coupling_degree': 0.704889}, {'task_id': 55, 'name': 'task_55', 'story_points': 2, 'req_skill': 'Backend', 'coupling_degree': 0.872608}, {'task_id': 56, 'name': 'task_56', 'story_points': 13, 'req_skill': 'Integration', 'coupling_degree': 0.228798}, {'task_id': 57, 'name': 'task_57', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.745348}, {'task_id': 58, 'name': 'task_58', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.653818}, {'task_id': 59, 'name': 'task_59', 'story_points': 2, 'req_skill': 'Backend', 'coupling_degree': 0.623251}, {'task_id': 60, 'name': 'task_60', 'story_points': 5, 'req_skill': 'Backend', 'coupling_degree': 0.578201}, {'task_id': 61, 'name': 'task_61', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.5}, {'task_id': 62, 'name': 'task_62', 'story_points': 13, 'req_skill': 'Backend', 'coupling_degree': 0.610088}, {'task_id': 63, 'name': 'task_63', 'story_points': 2, 'req_skill': 'UI/UX', 'coupling_degree': 0.721306}, {'task_id': 64, 'name': 'task_64', 'story_points': 2, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 65, 'name': 'task_65', 'story_points': 8, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 66, 'name': 'task_66', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.892132}, {'task_id': 67, 'name': 'task_67', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.570727}, {'task_id': 68, 'name': 'task_68', 'story_points': 1, 'req_skill': 'Integration', 'coupling_degree': 0.767425}, {'task_id': 69, 'name': 'task_69', 'story_points': 5, 'req_skill': 'Integration', 'coupling_degree': 0.79035}, {'task_id': 70, 'name': 'task_70', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 71, 'name': 'task_71', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 72, 'name': 'task_72', 'story_points': 1, 'req_skill': 'DB', 'coupling_degree': 0.863003}, {'task_id': 73, 'name': 'task_73', 'story_points': 5, 'req_skill': 'API', 'coupling_degree': 0.523212}, {'task_id': 74, 'name': 'task_74', 'story_points': 3, 'req_skill': 'DB', 'coupling_degree': 0.547173}, {'task_id': 75, 'name': 'task_75', 'story_points': 13, 'req_skill': 'UI/UX', 'coupling_degree': 0.530998}, {'task_id': 76, 'name': 'task_76', 'story_points': 8, 'req_skill': 'UI/UX', 'coupling_degree': 0.712788}, {'task_id': 77, 'name': 'task_77', 'story_points': 2, 'req_skill': 'UI/UX', 'coupling_degree': 0.690076}, {'task_id': 78, 'name': 'task_78', 'story_points': 2, 'req_skill': 'Backend', 'coupling_degree': 0.131936}, {'task_id': 79, 'name': 'task_79', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.460055}, {'task_id': 80, 'name': 'task_80', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.515658}, {'task_id': 81, 'name': 'task_81', 'story_points': 13, 'req_skill': 'API', 'coupling_degree': 0.748082}, {'task_id': 82, 'name': 'task_82', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.564923}, {'task_id': 83, 'name': 'task_83', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.773193}, {'task_id': 84, 'name': 'task_84', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 85, 'name': 'task_85', 'story_points': 8, 'req_skill': 'API', 'coupling_degree': 0.732843}, {'task_id': 86, 'name': 'task_86', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.68264}, {'task_id': 87, 'name': 'task_87', 'story_points': 13, 'req_skill': 'Backend', 'coupling_degree': 0.5}, {'task_id': 88, 'name': 'task_88', 'story_points': 13, 'req_skill': 'UI/UX', 'coupling_degree': 0.742058}, {'task_id': 89, 'name': 'task_89', 'story_points': 5, 'req_skill': 'Backend', 'coupling_degree': 0.5}, {'task_id': 90, 'name': 'task_90', 'story_points': 1, 'req_skill': 'DB', 'coupling_degree': 0.865322}, {'task_id': 91, 'name': 'task_91', 'story_points': 13, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 92, 'name': 'task_92', 'story_points': 3, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 93, 'name': 'task_93', 'story_points': 2, 'req_skill': 'DB', 'coupling_degree': 0.662385}, {'task_id': 94, 'name': 'task_94', 'story_points': 5, 'req_skill': 'Backend', 'coupling_degree': 0.895833}, {'task_id': 95, 'name': 'task_95', 'story_points': 3, 'req_skill': 'DB', 'coupling_degree': 0.436045}, {'task_id': 96, 'name': 'task_96', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.219486}, {'task_id': 97, 'name': 'task_97', 'story_points': 8, 'req_skill': 'UI/UX', 'coupling_degree': 0.856056}, {'task_id': 98, 'name': 'task_98', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.429702}, {'task_id': 99, 'name': 'task_99', 'story_points': 2, 'req_skill': 'Backend', 'coupling_degree': 0.534663}, {'task_id': 100, 'name': 'task_100', 'story_points': 3, 'req_skill': 'Integration', 'coupling_degree': 0.605297}, {'task_id': 101, 'name': 'task_101', 'story_points': 2, 'req_skill': 'Backend', 'coupling_degree': 0.5}, {'task_id': 102, 'name': 'task_102', 'story_points': 2, 'req_skill': 'Integration', 'coupling_degree': 0.673758}, {'task_id': 103, 'name': 'task_103', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.608318}, {'task_id': 104, 'name': 'task_104', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.899183}, {'task_id': 105, 'name': 'task_105', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.582951}, {'task_id': 106, 'name': 'task_106', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.222038}, {'task_id': 107, 'name': 'task_107', 'story_points': 1, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 108, 'name': 'task_108', 'story_points': 2, 'req_skill': 'Integration', 'coupling_degree': 0.5}, {'task_id': 109, 'name': 'task_109', 'story_points': 2, 'req_skill': 'Integration', 'coupling_degree': 0.747182}, {'task_id': 110, 'name': 'task_110', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.776436}, {'task_id': 111, 'name': 'task_111', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.585858}, {'task_id': 112, 'name': 'task_112', 'story_points': 3, 'req_skill': 'UI/UX', 'coupling_degree': 0.885011}, {'task_id': 113, 'name': 'task_113', 'story_points': 13, 'req_skill': 'Backend', 'coupling_degree': 0.51827}, {'task_id': 114, 'name': 'task_114', 'story_points': 8, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 115, 'name': 'task_115', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 116, 'name': 'task_116', 'story_points': 5, 'req_skill': 'Integration', 'coupling_degree': 0.5}, {'task_id': 117, 'name': 'task_117', 'story_points': 5, 'req_skill': 'API', 'coupling_degree': 0.71348}, {'task_id': 118, 'name': 'task_118', 'story_points': 5, 'req_skill': 'Backend', 'coupling_degree': 0.508392}, {'task_id': 119, 'name': 'task_119', 'story_points': 1, 'req_skill': 'API', 'coupling_degree': 0.88293}, {'task_id': 120, 'name': 'task_120', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.710347}, {'task_id': 121, 'name': 'task_121', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.602035}, {'task_id': 122, 'name': 'task_122', 'story_points': 13, 'req_skill': 'Integration', 'coupling_degree': 0.185407}, {'task_id': 123, 'name': 'task_123', 'story_points': 8, 'req_skill': 'DB', 'coupling_degree': 0.747638}, {'task_id': 124, 'name': 'task_124', 'story_points': 2, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 125, 'name': 'task_125', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.616918}, {'task_id': 126, 'name': 'task_126', 'story_points': 3, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 127, 'name': 'task_127', 'story_points': 1, 'req_skill': 'Backend', 'coupling_degree': 0.78983}, {'task_id': 128, 'name': 'task_128', 'story_points': 5, 'req_skill': 'Integration', 'coupling_degree': 0.624496}, {'task_id': 129, 'name': 'task_129', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.5}, {'task_id': 130, 'name': 'task_130', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.657001}, {'task_id': 131, 'name': 'task_131', 'story_points': 5, 'req_skill': 'Backend', 'coupling_degree': 0.553998}, {'task_id': 132, 'name': 'task_132', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.5}, {'task_id': 133, 'name': 'task_133', 'story_points': 3, 'req_skill': 'Integration', 'coupling_degree': 0.663825}, {'task_id': 134, 'name': 'task_134', 'story_points': 5, 'req_skill': 'Backend', 'coupling_degree': 0.231889}, {'task_id': 135, 'name': 'task_135', 'story_points': 2, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 136, 'name': 'task_136', 'story_points': 5, 'req_skill': 'API', 'coupling_degree': 0.650395}, {'task_id': 137, 'name': 'task_137', 'story_points': 5, 'req_skill': 'Integration', 'coupling_degree': 0.5}, {'task_id': 138, 'name': 'task_138', 'story_points': 3, 'req_skill': 'UI/UX', 'coupling_degree': 0.545286}, {'task_id': 139, 'name': 'task_139', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.72746}, {'task_id': 140, 'name': 'task_140', 'story_points': 5, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 141, 'name': 'task_141', 'story_points': 13, 'req_skill': 'UI/UX', 'coupling_degree': 0.888532}, {'task_id': 142, 'name': 'task_142', 'story_points': 2, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 143, 'name': 'task_143', 'story_points': 1, 'req_skill': 'Integration', 'coupling_degree': 0.5}, {'task_id': 144, 'name': 'task_144', 'story_points': 2, 'req_skill': 'UI/UX', 'coupling_degree': 0.539907}, {'task_id': 145, 'name': 'task_145', 'story_points': 5, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 146, 'name': 'task_146', 'story_points': 5, 'req_skill': 'Backend', 'coupling_degree': 0.5}, {'task_id': 147, 'name': 'task_147', 'story_points': 13, 'req_skill': 'Integration', 'coupling_degree': 0.424557}, {'task_id': 148, 'name': 'task_148', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.677102}, {'task_id': 149, 'name': 'task_149', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.589993}, {'task_id': 150, 'name': 'task_150', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.637827}, {'task_id': 151, 'name': 'task_151', 'story_points': 13, 'req_skill': 'API', 'coupling_degree': 0.720948}, {'task_id': 152, 'name': 'task_152', 'story_points': 13, 'req_skill': 'API', 'coupling_degree': 0.816458}, {'task_id': 153, 'name': 'task_153', 'story_points': 2, 'req_skill': 'API', 'coupling_degree': 0.287511}, {'task_id': 154, 'name': 'task_154', 'story_points': 2, 'req_skill': 'Backend', 'coupling_degree': 0.827354}, {'task_id': 155, 'name': 'task_155', 'story_points': 1, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 156, 'name': 'task_156', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.672314}, {'task_id': 157, 'name': 'task_157', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.713755}, {'task_id': 158, 'name': 'task_158', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.570866}, {'task_id': 159, 'name': 'task_159', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.670718}, {'task_id': 160, 'name': 'task_160', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.71789}, {'task_id': 161, 'name': 'task_161', 'story_points': 2, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 162, 'name': 'task_162', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 163, 'name': 'task_163', 'story_points': 2, 'req_skill': 'Integration', 'coupling_degree': 0.826891}, {'task_id': 164, 'name': 'task_164', 'story_points': 2, 'req_skill': 'Integration', 'coupling_degree': 0.529126}, {'task_id': 165, 'name': 'task_165', 'story_points': 8, 'req_skill': 'Backend', 'coupling_degree': 0.5}, {'task_id': 166, 'name': 'task_166', 'story_points': 1, 'req_skill': 'DB', 'coupling_degree': 0.778999}, {'task_id': 167, 'name': 'task_167', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.639051}, {'task_id': 168, 'name': 'task_168', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.761961}, {'task_id': 169, 'name': 'task_169', 'story_points': 8, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 170, 'name': 'task_170', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.530928}, {'task_id': 171, 'name': 'task_171', 'story_points': 2, 'req_skill': 'API', 'coupling_degree': 0.707593}, {'task_id': 172, 'name': 'task_172', 'story_points': 2, 'req_skill': 'UI/UX', 'coupling_degree': 0.523333}, {'task_id': 173, 'name': 'task_173', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 174, 'name': 'task_174', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.58408}, {'task_id': 175, 'name': 'task_175', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 176, 'name': 'task_176', 'story_points': 5, 'req_skill': 'API', 'coupling_degree': 0.577697}, {'task_id': 177, 'name': 'task_177', 'story_points': 13, 'req_skill': 'DB', 'coupling_degree': 0.636337}, {'task_id': 178, 'name': 'task_178', 'story_points': 13, 'req_skill': 'Integration', 'coupling_degree': 0.524549}, {'task_id': 179, 'name': 'task_179', 'story_points': 2, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 180, 'name': 'task_180', 'story_points': 8, 'req_skill': 'API', 'coupling_degree': 0.889591}, {'task_id': 181, 'name': 'task_181', 'story_points': 8, 'req_skill': 'DB', 'coupling_degree': 0.499517}, {'task_id': 182, 'name': 'task_182', 'story_points': 8, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 183, 'name': 'task_183', 'story_points': 8, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 184, 'name': 'task_184', 'story_points': 3, 'req_skill': 'Backend', 'coupling_degree': 0.650097}, {'task_id': 185, 'name': 'task_185', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.740011}, {'task_id': 186, 'name': 'task_186', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.612908}, {'task_id': 187, 'name': 'task_187', 'story_points': 8, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 188, 'name': 'task_188', 'story_points': 2, 'req_skill': 'DB', 'coupling_degree': 0.5}, {'task_id': 189, 'name': 'task_189', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.624398}, {'task_id': 190, 'name': 'task_190', 'story_points': 5, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 191, 'name': 'task_191', 'story_points': 13, 'req_skill': 'UI/UX', 'coupling_degree': 0.5}, {'task_id': 192, 'name': 'task_192', 'story_points': 1, 'req_skill': 'Integration', 'coupling_degree': 0.717142}, {'task_id': 193, 'name': 'task_193', 'story_points': 1, 'req_skill': 'DB', 'coupling_degree': 0.682427}, {'task_id': 194, 'name': 'task_194', 'story_points': 1, 'req_skill': 'UI/UX', 'coupling_degree': 0.793166}, {'task_id': 195, 'name': 'task_195', 'story_points': 3, 'req_skill': 'UI/UX', 'coupling_degree': 0.853554}, {'task_id': 196, 'name': 'task_196', 'story_points': 8, 'req_skill': 'Integration', 'coupling_degree': 0.635405}, {'task_id': 197, 'name': 'task_197', 'story_points': 3, 'req_skill': 'API', 'coupling_degree': 0.5}, {'task_id': 198, 'name': 'task_198', 'story_points': 8, 'req_skill': 'API', 'coupling_degree': 0.695184}, {'task_id': 199, 'name': 'task_199', 'story_points': 5, 'req_skill': 'DB', 'coupling_degree': 0.803685}]

RAW_DEPENDENCIES = [{'pred_id': 75, 'succ_id': 34, 'dep_type': 'cyclic', 'coupling_degree': 0.530998}, {'pred_id': 34, 'succ_id': 75, 'dep_type': 'cyclic', 'coupling_degree': 0.84763}, {'pred_id': 176, 'succ_id': 100, 'dep_type': 'cyclic', 'coupling_degree': 0.577697}, {'pred_id': 100, 'succ_id': 176, 'dep_type': 'cyclic', 'coupling_degree': 0.605297}, {'pred_id': 14, 'succ_id': 97, 'dep_type': 'cyclic', 'coupling_degree': 0.62332}, {'pred_id': 97, 'succ_id': 14, 'dep_type': 'cyclic', 'coupling_degree': 0.856056}, {'pred_id': 125, 'succ_id': 127, 'dep_type': 'cyclic', 'coupling_degree': 0.616918}, {'pred_id': 127, 'succ_id': 125, 'dep_type': 'cyclic', 'coupling_degree': 0.78983}, {'pred_id': 110, 'succ_id': 153, 'dep_type': 'cyclic', 'coupling_degree': 0.776436}, {'pred_id': 153, 'succ_id': 110, 'dep_type': 'cyclic', 'coupling_degree': 0.287511}, {'pred_id': 123, 'succ_id': 198, 'dep_type': 'cyclic', 'coupling_degree': 0.747638}, {'pred_id': 198, 'succ_id': 123, 'dep_type': 'cyclic', 'coupling_degree': 0.695184}, {'pred_id': 59, 'succ_id': 139, 'dep_type': 'cyclic', 'coupling_degree': 0.623251}, {'pred_id': 139, 'succ_id': 59, 'dep_type': 'cyclic', 'coupling_degree': 0.72746}, {'pred_id': 54, 'succ_id': 23, 'dep_type': 'cyclic', 'coupling_degree': 0.704889}, {'pred_id': 23, 'succ_id': 54, 'dep_type': 'cyclic', 'coupling_degree': 0.895222}, {'pred_id': 27, 'succ_id': 106, 'dep_type': 'cyclic', 'coupling_degree': 0.708127}, {'pred_id': 106, 'succ_id': 27, 'dep_type': 'cyclic', 'coupling_degree': 0.222038}, {'pred_id': 8, 'succ_id': 167, 'dep_type': 'cyclic', 'coupling_degree': 0.561801}, {'pred_id': 167, 'succ_id': 180, 'dep_type': 'cyclic', 'coupling_degree': 0.639051}, {'pred_id': 180, 'succ_id': 8, 'dep_type': 'cyclic', 'coupling_degree': 0.889591}, {'pred_id': 25, 'succ_id': 32, 'dep_type': 'cyclic', 'coupling_degree': 0.573302}, {'pred_id': 32, 'succ_id': 95, 'dep_type': 'cyclic', 'coupling_degree': 0.762248}, {'pred_id': 95, 'succ_id': 25, 'dep_type': 'cyclic', 'coupling_degree': 0.436045}, {'pred_id': 7, 'succ_id': 194, 'dep_type': 'acyclic', 'coupling_degree': 0.672417}, {'pred_id': 111, 'succ_id': 1, 'dep_type': 'acyclic', 'coupling_degree': 0.585858}, {'pred_id': 178, 'succ_id': 83, 'dep_type': 'acyclic', 'coupling_degree': 0.636913}, {'pred_id': 45, 'succ_id': 148, 'dep_type': 'acyclic', 'coupling_degree': 0.813437}, {'pred_id': 148, 'succ_id': 145, 'dep_type': 'acyclic', 'coupling_degree': 0.677102}, {'pred_id': 43, 'succ_id': 113, 'dep_type': 'acyclic', 'coupling_degree': 0.65522}, {'pred_id': 48, 'succ_id': 105, 'dep_type': 'acyclic', 'coupling_degree': 0.628752}, {'pred_id': 118, 'succ_id': 104, 'dep_type': 'acyclic', 'coupling_degree': 0.700829}, {'pred_id': 38, 'succ_id': 89, 'dep_type': 'acyclic', 'coupling_degree': 0.867404}, {'pred_id': 170, 'succ_id': 64, 'dep_type': 'acyclic', 'coupling_degree': 0.530928}, {'pred_id': 151, 'succ_id': 173, 'dep_type': 'acyclic', 'coupling_degree': 0.720948}, {'pred_id': 35, 'succ_id': 65, 'dep_type': 'acyclic', 'coupling_degree': 0.729319}, {'pred_id': 85, 'succ_id': 120, 'dep_type': 'acyclic', 'coupling_degree': 0.702811}, {'pred_id': 117, 'succ_id': 70, 'dep_type': 'acyclic', 'coupling_degree': 0.669025}, {'pred_id': 12, 'succ_id': 94, 'dep_type': 'acyclic', 'coupling_degree': 0.747377}, {'pred_id': 73, 'succ_id': 108, 'dep_type': 'acyclic', 'coupling_degree': 0.523212}, {'pred_id': 30, 'succ_id': 140, 'dep_type': 'acyclic', 'coupling_degree': 0.817019}, {'pred_id': 105, 'succ_id': 189, 'dep_type': 'acyclic', 'coupling_degree': 0.582951}, {'pred_id': 164, 'succ_id': 46, 'dep_type': 'acyclic', 'coupling_degree': 0.642481}, {'pred_id': 50, 'succ_id': 178, 'dep_type': 'acyclic', 'coupling_degree': 0.770295}, {'pred_id': 93, 'succ_id': 160, 'dep_type': 'acyclic', 'coupling_degree': 0.488886}, {'pred_id': 96, 'succ_id': 119, 'dep_type': 'acyclic', 'coupling_degree': 0.163536}, {'pred_id': 80, 'succ_id': 160, 'dep_type': 'acyclic', 'coupling_degree': 0.514626}, {'pred_id': 82, 'succ_id': 161, 'dep_type': 'acyclic', 'coupling_degree': 0.623303}, {'pred_id': 166, 'succ_id': 12, 'dep_type': 'acyclic', 'coupling_degree': 0.778999}, {'pred_id': 58, 'succ_id': 135, 'dep_type': 'acyclic', 'coupling_degree': 0.895299}, {'pred_id': 80, 'succ_id': 48, 'dep_type': 'acyclic', 'coupling_degree': 0.794995}, {'pred_id': 174, 'succ_id': 119, 'dep_type': 'acyclic', 'coupling_degree': 0.651196}, {'pred_id': 158, 'succ_id': 83, 'dep_type': 'acyclic', 'coupling_degree': 0.84964}, {'pred_id': 113, 'succ_id': 137, 'dep_type': 'acyclic', 'coupling_degree': 0.885028}, {'pred_id': 11, 'succ_id': 101, 'dep_type': 'acyclic', 'coupling_degree': 0.597342}, {'pred_id': 22, 'succ_id': 147, 'dep_type': 'acyclic', 'coupling_degree': 0.777235}, {'pred_id': 152, 'succ_id': 67, 'dep_type': 'acyclic', 'coupling_degree': 0.431429}, {'pred_id': 121, 'succ_id': 114, 'dep_type': 'acyclic', 'coupling_degree': 0.602035}, {'pred_id': 174, 'succ_id': 99, 'dep_type': 'acyclic', 'coupling_degree': 0.602329}, {'pred_id': 98, 'succ_id': 63, 'dep_type': 'acyclic', 'coupling_degree': 0.429702}, {'pred_id': 42, 'succ_id': 128, 'dep_type': 'acyclic', 'coupling_degree': 0.754303}, {'pred_id': 3, 'succ_id': 49, 'dep_type': 'acyclic', 'coupling_degree': 0.595842}, {'pred_id': 2, 'succ_id': 190, 'dep_type': 'acyclic', 'coupling_degree': 0.648533}, {'pred_id': 118, 'succ_id': 101, 'dep_type': 'acyclic', 'coupling_degree': 0.7734}, {'pred_id': 33, 'succ_id': 140, 'dep_type': 'acyclic', 'coupling_degree': 0.784759}, {'pred_id': 90, 'succ_id': 119, 'dep_type': 'acyclic', 'coupling_degree': 0.865322}, {'pred_id': 192, 'succ_id': 141, 'dep_type': 'acyclic', 'coupling_degree': 0.571226}, {'pred_id': 76, 'succ_id': 151, 'dep_type': 'acyclic', 'coupling_degree': 0.654344}, {'pred_id': 154, 'succ_id': 120, 'dep_type': 'acyclic', 'coupling_degree': 0.827354}, {'pred_id': 20, 'succ_id': 152, 'dep_type': 'acyclic', 'coupling_degree': 0.590898}, {'pred_id': 199, 'succ_id': 148, 'dep_type': 'acyclic', 'coupling_degree': 0.550013}, {'pred_id': 172, 'succ_id': 109, 'dep_type': 'acyclic', 'coupling_degree': 0.605377}, {'pred_id': 128, 'succ_id': 166, 'dep_type': 'acyclic', 'coupling_degree': 0.624496}, {'pred_id': 118, 'succ_id': 140, 'dep_type': 'acyclic', 'coupling_degree': 0.508392}, {'pred_id': 68, 'succ_id': 62, 'dep_type': 'acyclic', 'coupling_degree': 0.837764}, {'pred_id': 55, 'succ_id': 120, 'dep_type': 'acyclic', 'coupling_degree': 0.872608}, {'pred_id': 181, 'succ_id': 80, 'dep_type': 'acyclic', 'coupling_degree': 0.499517}, {'pred_id': 185, 'succ_id': 172, 'dep_type': 'acyclic', 'coupling_degree': 0.740011}, {'pred_id': 29, 'succ_id': 107, 'dep_type': 'acyclic', 'coupling_degree': 0.642592}, {'pred_id': 147, 'succ_id': 64, 'dep_type': 'acyclic', 'coupling_degree': 0.424557}, {'pred_id': 194, 'succ_id': 158, 'dep_type': 'acyclic', 'coupling_degree': 0.687028}, {'pred_id': 113, 'succ_id': 26, 'dep_type': 'acyclic', 'coupling_degree': 0.51827}, {'pred_id': 82, 'succ_id': 7, 'dep_type': 'acyclic', 'coupling_degree': 0.564923}, {'pred_id': 93, 'succ_id': 33, 'dep_type': 'acyclic', 'coupling_degree': 0.662385}, {'pred_id': 50, 'succ_id': 16, 'dep_type': 'acyclic', 'coupling_degree': 0.561498}, {'pred_id': 174, 'succ_id': 13, 'dep_type': 'acyclic', 'coupling_degree': 0.610076}, {'pred_id': 19, 'succ_id': 89, 'dep_type': 'acyclic', 'coupling_degree': 0.680119}, {'pred_id': 60, 'succ_id': 171, 'dep_type': 'acyclic', 'coupling_degree': 0.763071}, {'pred_id': 18, 'succ_id': 67, 'dep_type': 'acyclic', 'coupling_degree': 0.671389}, {'pred_id': 192, 'succ_id': 131, 'dep_type': 'acyclic', 'coupling_degree': 0.506813}, {'pred_id': 174, 'succ_id': 196, 'dep_type': 'acyclic', 'coupling_degree': 0.554636}, {'pred_id': 80, 'succ_id': 103, 'dep_type': 'acyclic', 'coupling_degree': 0.515658}, {'pred_id': 9, 'succ_id': 132, 'dep_type': 'acyclic', 'coupling_degree': 0.852309}, {'pred_id': 20, 'succ_id': 62, 'dep_type': 'acyclic', 'coupling_degree': 0.881586}, {'pred_id': 79, 'succ_id': 175, 'dep_type': 'acyclic', 'coupling_degree': 0.460055}, {'pred_id': 38, 'succ_id': 49, 'dep_type': 'acyclic', 'coupling_degree': 0.619678}, {'pred_id': 74, 'succ_id': 185, 'dep_type': 'acyclic', 'coupling_degree': 0.106747}, {'pred_id': 99, 'succ_id': 72, 'dep_type': 'acyclic', 'coupling_degree': 0.52538}, {'pred_id': 186, 'succ_id': 15, 'dep_type': 'acyclic', 'coupling_degree': 0.612908}, {'pred_id': 112, 'succ_id': 142, 'dep_type': 'acyclic', 'coupling_degree': 0.885011}, {'pred_id': 18, 'succ_id': 182, 'dep_type': 'acyclic', 'coupling_degree': 0.558044}, {'pred_id': 159, 'succ_id': 39, 'dep_type': 'acyclic', 'coupling_degree': 0.583781}, {'pred_id': 88, 'succ_id': 45, 'dep_type': 'acyclic', 'coupling_degree': 0.742058}, {'pred_id': 102, 'succ_id': 131, 'dep_type': 'acyclic', 'coupling_degree': 0.673758}, {'pred_id': 133, 'succ_id': 179, 'dep_type': 'acyclic', 'coupling_degree': 0.840746}, {'pred_id': 49, 'succ_id': 83, 'dep_type': 'acyclic', 'coupling_degree': 0.171832}, {'pred_id': 41, 'succ_id': 30, 'dep_type': 'acyclic', 'coupling_degree': 0.82811}, {'pred_id': 144, 'succ_id': 126, 'dep_type': 'acyclic', 'coupling_degree': 0.539907}, {'pred_id': 68, 'succ_id': 65, 'dep_type': 'acyclic', 'coupling_degree': 0.767425}, {'pred_id': 74, 'succ_id': 61, 'dep_type': 'acyclic', 'coupling_degree': 0.547173}, {'pred_id': 56, 'succ_id': 158, 'dep_type': 'acyclic', 'coupling_degree': 0.815331}, {'pred_id': 168, 'succ_id': 131, 'dep_type': 'acyclic', 'coupling_degree': 0.761961}, {'pred_id': 52, 'succ_id': 194, 'dep_type': 'acyclic', 'coupling_degree': 0.819116}, {'pred_id': 83, 'succ_id': 115, 'dep_type': 'acyclic', 'coupling_degree': 0.651146}, {'pred_id': 133, 'succ_id': 148, 'dep_type': 'acyclic', 'coupling_degree': 0.663825}, {'pred_id': 83, 'succ_id': 98, 'dep_type': 'acyclic', 'coupling_degree': 0.615591}, {'pred_id': 86, 'succ_id': 72, 'dep_type': 'acyclic', 'coupling_degree': 0.68264}, {'pred_id': 39, 'succ_id': 67, 'dep_type': 'acyclic', 'coupling_degree': 0.774211}, {'pred_id': 171, 'succ_id': 118, 'dep_type': 'acyclic', 'coupling_degree': 0.536388}, {'pred_id': 195, 'succ_id': 170, 'dep_type': 'acyclic', 'coupling_degree': 0.853554}, {'pred_id': 15, 'succ_id': 31, 'dep_type': 'acyclic', 'coupling_degree': 0.583086}, {'pred_id': 141, 'succ_id': 88, 'dep_type': 'acyclic', 'coupling_degree': 0.742723}, {'pred_id': 136, 'succ_id': 77, 'dep_type': 'acyclic', 'coupling_degree': 0.650395}, {'pred_id': 57, 'succ_id': 131, 'dep_type': 'acyclic', 'coupling_degree': 0.516483}, {'pred_id': 50, 'succ_id': 134, 'dep_type': 'acyclic', 'coupling_degree': 0.524721}, {'pred_id': 103, 'succ_id': 46, 'dep_type': 'acyclic', 'coupling_degree': 0.608318}, {'pred_id': 130, 'succ_id': 47, 'dep_type': 'acyclic', 'coupling_degree': 0.577984}, {'pred_id': 13, 'succ_id': 151, 'dep_type': 'acyclic', 'coupling_degree': 0.85433}, {'pred_id': 56, 'succ_id': 61, 'dep_type': 'acyclic', 'coupling_degree': 0.757638}, {'pred_id': 50, 'succ_id': 40, 'dep_type': 'acyclic', 'coupling_degree': 0.808911}, {'pred_id': 94, 'succ_id': 30, 'dep_type': 'acyclic', 'coupling_degree': 0.895833}, {'pred_id': 72, 'succ_id': 187, 'dep_type': 'acyclic', 'coupling_degree': 0.863003}, {'pred_id': 69, 'succ_id': 10, 'dep_type': 'acyclic', 'coupling_degree': 0.479025}, {'pred_id': 178, 'succ_id': 199, 'dep_type': 'acyclic', 'coupling_degree': 0.883115}, {'pred_id': 149, 'succ_id': 177, 'dep_type': 'acyclic', 'coupling_degree': 0.589993}, {'pred_id': 17, 'succ_id': 109, 'dep_type': 'acyclic', 'coupling_degree': 0.58702}, {'pred_id': 193, 'succ_id': 107, 'dep_type': 'acyclic', 'coupling_degree': 0.682427}, {'pred_id': 28, 'succ_id': 44, 'dep_type': 'acyclic', 'coupling_degree': 0.694075}, {'pred_id': 150, 'succ_id': 183, 'dep_type': 'acyclic', 'coupling_degree': 0.637827}, {'pred_id': 58, 'succ_id': 76, 'dep_type': 'acyclic', 'coupling_degree': 0.653818}, {'pred_id': 76, 'succ_id': 72, 'dep_type': 'acyclic', 'coupling_degree': 0.762779}, {'pred_id': 81, 'succ_id': 118, 'dep_type': 'acyclic', 'coupling_degree': 0.730019}, {'pred_id': 96, 'succ_id': 41, 'dep_type': 'acyclic', 'coupling_degree': 0.219486}, {'pred_id': 81, 'succ_id': 194, 'dep_type': 'acyclic', 'coupling_degree': 0.748082}, {'pred_id': 196, 'succ_id': 195, 'dep_type': 'acyclic', 'coupling_degree': 0.623622}, {'pred_id': 56, 'succ_id': 33, 'dep_type': 'acyclic', 'coupling_degree': 0.228798}, {'pred_id': 171, 'succ_id': 184, 'dep_type': 'acyclic', 'coupling_degree': 0.707593}, {'pred_id': 66, 'succ_id': 42, 'dep_type': 'acyclic', 'coupling_degree': 0.675542}, {'pred_id': 17, 'succ_id': 121, 'dep_type': 'acyclic', 'coupling_degree': 0.564216}, {'pred_id': 49, 'succ_id': 155, 'dep_type': 'acyclic', 'coupling_degree': 0.658303}, {'pred_id': 85, 'succ_id': 140, 'dep_type': 'acyclic', 'coupling_degree': 0.547347}, {'pred_id': 134, 'succ_id': 84, 'dep_type': 'acyclic', 'coupling_degree': 0.389881}, {'pred_id': 159, 'succ_id': 98, 'dep_type': 'acyclic', 'coupling_degree': 0.670718}, {'pred_id': 51, 'succ_id': 61, 'dep_type': 'acyclic', 'coupling_degree': 0.703473}, {'pred_id': 138, 'succ_id': 194, 'dep_type': 'acyclic', 'coupling_degree': 0.545286}, {'pred_id': 51, 'succ_id': 108, 'dep_type': 'acyclic', 'coupling_degree': 0.432661}, {'pred_id': 134, 'succ_id': 112, 'dep_type': 'acyclic', 'coupling_degree': 0.231889}, {'pred_id': 10, 'succ_id': 22, 'dep_type': 'acyclic', 'coupling_degree': 0.8262}, {'pred_id': 4, 'succ_id': 173, 'dep_type': 'acyclic', 'coupling_degree': 0.155573}, {'pred_id': 85, 'succ_id': 5, 'dep_type': 'acyclic', 'coupling_degree': 0.732843}, {'pred_id': 160, 'succ_id': 50, 'dep_type': 'acyclic', 'coupling_degree': 0.798091}, {'pred_id': 76, 'succ_id': 0, 'dep_type': 'acyclic', 'coupling_degree': 0.712788}, {'pred_id': 99, 'succ_id': 2, 'dep_type': 'acyclic', 'coupling_degree': 0.534663}, {'pred_id': 9, 'succ_id': 57, 'dep_type': 'acyclic', 'coupling_degree': 0.639216}, {'pred_id': 120, 'succ_id': 26, 'dep_type': 'acyclic', 'coupling_degree': 0.710347}, {'pred_id': 24, 'succ_id': 39, 'dep_type': 'acyclic', 'coupling_degree': 0.663611}, {'pred_id': 119, 'succ_id': 28, 'dep_type': 'acyclic', 'coupling_degree': 0.88293}, {'pred_id': 52, 'succ_id': 196, 'dep_type': 'acyclic', 'coupling_degree': 0.63932}, {'pred_id': 141, 'succ_id': 65, 'dep_type': 'acyclic', 'coupling_degree': 0.888532}, {'pred_id': 157, 'succ_id': 86, 'dep_type': 'acyclic', 'coupling_degree': 0.713755}, {'pred_id': 177, 'succ_id': 51, 'dep_type': 'acyclic', 'coupling_degree': 0.636337}, {'pred_id': 43, 'succ_id': 144, 'dep_type': 'acyclic', 'coupling_degree': 0.668434}, {'pred_id': 189, 'succ_id': 194, 'dep_type': 'acyclic', 'coupling_degree': 0.624398}, {'pred_id': 30, 'succ_id': 55, 'dep_type': 'acyclic', 'coupling_degree': 0.731283}, {'pred_id': 131, 'succ_id': 45, 'dep_type': 'acyclic', 'coupling_degree': 0.553998}, {'pred_id': 164, 'succ_id': 182, 'dep_type': 'acyclic', 'coupling_degree': 0.529126}, {'pred_id': 40, 'succ_id': 199, 'dep_type': 'acyclic', 'coupling_degree': 0.850122}, {'pred_id': 36, 'succ_id': 130, 'dep_type': 'acyclic', 'coupling_degree': 0.592528}, {'pred_id': 117, 'succ_id': 191, 'dep_type': 'acyclic', 'coupling_degree': 0.71348}, {'pred_id': 178, 'succ_id': 20, 'dep_type': 'acyclic', 'coupling_degree': 0.524549}, {'pred_id': 63, 'succ_id': 116, 'dep_type': 'acyclic', 'coupling_degree': 0.721306}, {'pred_id': 130, 'succ_id': 72, 'dep_type': 'acyclic', 'coupling_degree': 0.657001}, {'pred_id': 172, 'succ_id': 13, 'dep_type': 'acyclic', 'coupling_degree': 0.523333}, {'pred_id': 192, 'succ_id': 69, 'dep_type': 'acyclic', 'coupling_degree': 0.717142}, {'pred_id': 160, 'succ_id': 148, 'dep_type': 'acyclic', 'coupling_degree': 0.71789}, {'pred_id': 174, 'succ_id': 154, 'dep_type': 'acyclic', 'coupling_degree': 0.58408}, {'pred_id': 194, 'succ_id': 144, 'dep_type': 'acyclic', 'coupling_degree': 0.793166}, {'pred_id': 9, 'succ_id': 150, 'dep_type': 'acyclic', 'coupling_degree': 0.60147}, {'pred_id': 4, 'succ_id': 86, 'dep_type': 'acyclic', 'coupling_degree': 0.576794}, {'pred_id': 109, 'succ_id': 196, 'dep_type': 'acyclic', 'coupling_degree': 0.747182}, {'pred_id': 158, 'succ_id': 98, 'dep_type': 'acyclic', 'coupling_degree': 0.570866}, {'pred_id': 46, 'succ_id': 45, 'dep_type': 'acyclic', 'coupling_degree': 0.812555}, {'pred_id': 67, 'succ_id': 90, 'dep_type': 'acyclic', 'coupling_degree': 0.570727}, {'pred_id': 152, 'succ_id': 30, 'dep_type': 'acyclic', 'coupling_degree': 0.816458}, {'pred_id': 78, 'succ_id': 158, 'dep_type': 'acyclic', 'coupling_degree': 0.131936}, {'pred_id': 15, 'succ_id': 30, 'dep_type': 'acyclic', 'coupling_degree': 0.879063}, {'pred_id': 51, 'succ_id': 63, 'dep_type': 'acyclic', 'coupling_degree': 0.515713}, {'pred_id': 199, 'succ_id': 150, 'dep_type': 'acyclic', 'coupling_degree': 0.519887}, {'pred_id': 44, 'succ_id': 92, 'dep_type': 'acyclic', 'coupling_degree': 0.522283}, {'pred_id': 21, 'succ_id': 121, 'dep_type': 'acyclic', 'coupling_degree': 0.623294}, {'pred_id': 104, 'succ_id': 182, 'dep_type': 'acyclic', 'coupling_degree': 0.899183}, {'pred_id': 66, 'succ_id': 151, 'dep_type': 'acyclic', 'coupling_degree': 0.892132}, {'pred_id': 156, 'succ_id': 103, 'dep_type': 'acyclic', 'coupling_degree': 0.672314}, {'pred_id': 163, 'succ_id': 146, 'dep_type': 'acyclic', 'coupling_degree': 0.826891}, {'pred_id': 83, 'succ_id': 15, 'dep_type': 'acyclic', 'coupling_degree': 0.773193}, {'pred_id': 43, 'succ_id': 138, 'dep_type': 'acyclic', 'coupling_degree': 0.762024}, {'pred_id': 40, 'succ_id': 48, 'dep_type': 'acyclic', 'coupling_degree': 0.337036}, {'pred_id': 122, 'succ_id': 83, 'dep_type': 'acyclic', 'coupling_degree': 0.185407}, {'pred_id': 33, 'succ_id': 91, 'dep_type': 'acyclic', 'coupling_degree': 0.574439}, {'pred_id': 199, 'succ_id': 88, 'dep_type': 'acyclic', 'coupling_degree': 0.803685}, {'pred_id': 49, 'succ_id': 74, 'dep_type': 'acyclic', 'coupling_degree': 0.851303}, {'pred_id': 196, 'succ_id': 140, 'dep_type': 'acyclic', 'coupling_degree': 0.635405}, {'pred_id': 36, 'succ_id': 84, 'dep_type': 'acyclic', 'coupling_degree': 0.745464}, {'pred_id': 184, 'succ_id': 37, 'dep_type': 'acyclic', 'coupling_degree': 0.650097}, {'pred_id': 52, 'succ_id': 20, 'dep_type': 'acyclic', 'coupling_degree': 0.74411}, {'pred_id': 62, 'succ_id': 141, 'dep_type': 'acyclic', 'coupling_degree': 0.610088}, {'pred_id': 69, 'succ_id': 64, 'dep_type': 'acyclic', 'coupling_degree': 0.79035}, {'pred_id': 57, 'succ_id': 89, 'dep_type': 'acyclic', 'coupling_degree': 0.745348}, {'pred_id': 77, 'succ_id': 149, 'dep_type': 'acyclic', 'coupling_degree': 0.690076}, {'pred_id': 60, 'succ_id': 63, 'dep_type': 'acyclic', 'coupling_degree': 0.578201}]

def load_manual_real_world_instance():
    teams = [Team(t['team_id'], t['name']) for t in RAW_TEAMS]
    devs = [Developer(d['dev_id'], d['name'], d['skills'], d['exp_level'], d['team_id']) for d in RAW_DEVELOPERS]
    tasks = [Task(tk['task_id'], tk['name'], tk['req_skill'], tk['story_points'], tk['coupling_degree']) for tk in RAW_TASKS]
    deps = [Dependency(dp['pred_id'], dp['succ_id'], dp['dep_type'], dp['coupling_degree']) for dp in RAW_DEPENDENCIES]
    return tasks, devs, teams, deps

real_tasks, real_devs, real_teams, real_deps = load_manual_real_world_instance()
print('=== Real-World Instance Manually Loaded ===')
print(f'Tasks: {len(real_tasks)} | Developers: {len(real_devs)} | Teams: {len(real_teams)} | Dependencies: {len(real_deps)}')


### 2. Dependency Graph Visualizer
Visualizes the circular dependency graph of the real-world project instance using `networkx`, displaying black single-way acyclic arrows and two-way straight red cyclic arrows (`<->`).

In [ ]:
def visualize_dependency_graph(tasks, dependencies, title_prefix=''):
    G = nx.DiGraph()
    for t in tasks:
        tid = t.task_id if hasattr(t, 'task_id') else t['id']
        G.add_node(tid)
    acyclic_edges = []
    cyclic_edges = []
    for dep in dependencies:
        p = dep.pred_id if hasattr(dep, 'pred_id') else dep['pred']
        s = dep.succ_id if hasattr(dep, 'succ_id') else dep['succ']
        dep_type = dep.dep_type if hasattr(dep, 'dep_type') else dep.get('type', 'acyclic')
        G.add_edge(p, s)
        if dep_type == 'cyclic':
            cyclic_edges.append((p, s))
        else:
            acyclic_edges.append((p, s))
    couplings = [t.coupling_degree if hasattr(t, 'coupling_degree') else t['coupling'] for t in tasks]
    avg_coupling = np.mean(couplings) if len(couplings) > 0 else 0.0
    plt.figure(figsize=(9, 8), dpi=85)
    pos = nx.circular_layout(G)
    node_size = 180 if len(tasks) > 50 else 400
    font_size = 5 if len(tasks) > 50 else 8
    nx.draw_networkx_nodes(G, pos, node_color='#BCE0FD', node_size=node_size, edgecolors='#333333', linewidths=0.8)
    nx.draw_networkx_labels(G, pos, font_size=font_size, font_family='sans-serif')
    if acyclic_edges:
        nx.draw_networkx_edges(G, pos, edgelist=acyclic_edges, edge_color='black', arrows=True, arrowstyle='-|>', arrowsize=10, width=0.7, alpha=0.6, connectionstyle='arc3,rad=0')
    if cyclic_edges:
        nx.draw_networkx_edges(G, pos, edgelist=cyclic_edges, edge_color='red', arrows=True, arrowstyle='<|-|>', arrowsize=14, width=1.4, connectionstyle='arc3,rad=0')
    black_line = plt.Line2D([], [], color='black', marker='>', markersize=6, label='Acyclic Dependencies')
    red_line = plt.Line2D([], [], color='red', marker='<', markersize=6, label='Cyclic Dependencies (Two-Way)')
    plt.legend(handles=[black_line, red_line], loc='upper right', fontsize=9)
    total_deps = len(dependencies)
    num_cyclic = len(cyclic_edges)
    title_text = f'Total Dependencies: {total_deps}, Cyclic: {num_cyclic}, Avg Coupling: {avg_coupling:.3f}'
    if title_prefix:
        title_text = f'{title_prefix}\n{title_text}'
    plt.title(title_text, fontsize=11, fontweight='bold', pad=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Visualize full 200-task real-world instance
# visualize_dependency_graph(real_tasks, real_deps, title_prefix='Real-World Dataset Instance (200 Tasks, 220 Dependencies)')


### 3. Define the SPSP Problem Class with Skill-Aware Decoder & Cyclic Risk Calculation
Subclasses `ElementwiseProblem` with $2N = 400$ continuous decision variables ($200$ developer assignments + $200$ start delays).

**Key Features:**
- **Skill-Aware Decoder (Option A):** Maps $x_i \to$ qualified developers possessing `req_skill(t_i)`. Guarantees 100% skill feasibility ($g_{\text{skill}, i} = 0.0$).
- **Coordination Breakdown Risk ($f_3$):** Calculates FMEA RPN risk ($R = S \times O \times D$) for cyclic re-engineering dependencies ($E_{\text{cyc}}$) and concurrent task overlaps across team boundaries.

In [ ]:
from pymoo.core.problem import ElementwiseProblem

class SoftwareProjectProblem(ElementwiseProblem):
    def __init__(self, tasks, developers, teams, dependencies, T_ref=1.0, alpha=0.7, beta=1.4, max_delay=10.0):
        n_tasks = len(tasks)
        n_devs = len(developers)
        n_constr = n_tasks
        xl = np.zeros(2 * n_tasks)
        xu = np.zeros(2 * n_tasks)
        xl[:n_tasks] = 0.0
        xu[:n_tasks] = float(n_devs - 1e-5)
        xl[n_tasks:] = 0.0
        xu[n_tasks:] = float(max_delay)
        super().__init__(n_var=2*n_tasks, n_obj=3, n_constr=n_constr, xl=xl, xu=xu)
        self.tasks = tasks
        self.developers = developers
        self.teams = teams
        self.dependencies = dependencies
        self.n_tasks = n_tasks
        self.n_devs = n_devs
        self.T_ref = T_ref
        self.alpha = alpha
        self.beta = beta
        
        self.qualified_devs_per_task = []
        for task in tasks:
            q_devs = [d for d in developers if task.req_skill in d.skills]
            if not q_devs:
                q_devs = developers
            self.qualified_devs_per_task.append(q_devs)

    def ad_just(self, exp_level: str) -> float:
        if exp_level == 'A':
            return self.alpha
        elif exp_level == 'B':
            return self.beta
        return 1.0

    def dev_time(self, task: Task, dev: Developer) -> float:
        return task.story_points * self.T_ref * self.ad_just(dev.exp_level)

    def _evaluate(self, x, out, *args, **kwargs):
        task_durations = np.zeros(self.n_tasks)
        assigned_devs = []
        g_skills = []
        
        for i in range(self.n_tasks):
            q_devs = self.qualified_devs_per_task[i]
            q_idx = int(np.floor((x[i] / self.n_devs) * len(q_devs))) % len(q_devs)
            dev = q_devs[q_idx]
            assigned_devs.append(dev)
            
            task = self.tasks[i]
            duration = self.dev_time(task, dev)
            task_durations[i] = duration
            g_skills.append(0.0)
            
        delays = x[self.n_tasks:]
        start_times = np.zeros(self.n_tasks)
        finish_times = np.zeros(self.n_tasks)
        dev_free_time = np.zeros(self.n_devs)
        pred_map = {i: [] for i in range(self.n_tasks)}
        for dep in self.dependencies:
            if dep.dep_type != 'cyclic':
                pred_map[dep.succ_id].append(dep.pred_id)
            
        for i in range(self.n_tasks):
            assigned_dev = assigned_devs[i]
            dur = task_durations[i]
            dep_ready = max([finish_times[p] for p in pred_map[i]], default=0.0)
            dev_ready = dev_free_time[assigned_dev.dev_id]
            earliest_start = max(dep_ready, dev_ready)
            start_i = earliest_start + delays[i]
            finish_i = start_i + dur
            start_times[i] = start_i
            finish_times[i] = finish_i
            dev_free_time[assigned_dev.dev_id] = finish_i
            
        f1_makespan = np.max(finish_times)
        dev_workloads = np.zeros(self.n_devs)
        for i in range(self.n_tasks):
            dev_workloads[assigned_devs[i].dev_id] += task_durations[i]
        f2_workload_imbalance = np.std(dev_workloads)
        
        # Calculate Coordination Breakdown Risk (f3)
        f3_coordination_risk = 0.0
        theta_overlap = 0.5
        for dep in self.dependencies:
            p, s = dep.pred_id, dep.succ_id
            dev_p = assigned_devs[p]
            dev_s = assigned_devs[s]
            if dev_p.dev_id != dev_s.dev_id:
                if dep.dep_type == 'cyclic':
                    occur = 1.5 if dev_p.team_id != dev_s.team_id else 1.0
                    detect = 2.0 if self.tasks[s].req_skill not in dev_p.skills else 1.0
                    sev = self.tasks[s].coupling_degree * task_durations[s]
                    f3_coordination_risk += sev * occur * detect
                else:
                    overlap = min(finish_times[p], finish_times[s]) - max(start_times[p], start_times[s])
                    if overlap >= theta_overlap:
                        occur = 1.5 if dev_p.team_id != dev_s.team_id else 1.0
                        detect = 2.0 if self.tasks[s].req_skill not in dev_p.skills else 1.0
                        sev = self.tasks[s].coupling_degree * task_durations[s]
                        f3_coordination_risk += sev * occur * detect
                    
        out['F'] = [f1_makespan, f2_workload_imbalance, f3_coordination_risk]
        out['G'] = g_skills

print('SoftwareProjectProblem class defined for Full 200 Tasks Instance.')


### 4. Run NSGA-II and MOPSO-CD on Full 200-Task Real-World Instance
Executes NSGA-II and MOPSO-CD on the **entire 200 tasks** and **220 dependencies** of the real-world dataset.

In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.mopso_cd import MOPSO_CD
from pymoo.optimize import minimize

eval_tasks = real_tasks
eval_deps = real_deps

problem = SoftwareProjectProblem(eval_tasks, real_devs, real_teams, eval_deps)
pop_size = 40
n_gen = 40

start_t = time.time()
nsga2_algo = NSGA2(pop_size=pop_size)
res_nsga2 = minimize(problem, nsga2_algo, ('n_gen', n_gen), seed=42, verbose=False)
t_nsga2 = time.time() - start_t

start_t = time.time()
mopso_algo = MOPSO_CD(pop_size=pop_size)
res_mopso = minimize(problem, mopso_algo, ('n_gen', n_gen), seed=42, verbose=False)
t_mopso = time.time() - start_t

print('=== Optimization Runs Completed on Full 200-Task Real-World Instance ===')
print(f'NSGA-II:  Runtime = {t_nsga2:.3f}s | Feasible Pareto Solutions = {len(res_nsga2.F) if res_nsga2.F is not None else 0}')
print(f'MOPSO-CD: Runtime = {t_mopso:.3f}s | Feasible Pareto Solutions = {len(res_mopso.F) if res_mopso.F is not None else 0}')


### 5. Visualization of Pareto Fronts & Efficiency
Plots 3D Pareto Front scatter plots, 2D pairwise objective projections, and performance bar charts for the full 200-task instance.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 5), dpi=85)
ax1 = fig.add_subplot(121, projection='3d')
if res_nsga2.F is not None:
    ax1.scatter(res_nsga2.F[:, 0], res_nsga2.F[:, 1], res_nsga2.F[:, 2], c='crimson', label='NSGA-II', s=40, alpha=0.8)
if res_mopso.F is not None:
    ax1.scatter(res_mopso.F[:, 0], res_mopso.F[:, 1], res_mopso.F[:, 2], c='teal', label='MOPSO-CD', s=40, alpha=0.8)
ax1.set_xlabel('Makespan ($f_1$)')
ax1.set_ylabel('Workload Imbalance ($f_2$)')
ax1.set_zlabel('Coordination Risk ($f_3$)')
ax1.set_title('3D Pareto Front Comparison')
ax1.legend()

ax2 = fig.add_subplot(122)
algos = ['NSGA-II', 'MOPSO-CD']
runtimes = [t_nsga2, t_mopso]
front_sizes = [len(res_nsga2.F) if res_nsga2.F is not None else 0, len(res_mopso.F) if res_mopso.F is not None else 0]
x_indices = np.arange(len(algos))
width = 0.35
rects1 = ax2.bar(x_indices - width/2, runtimes, width, label='Runtime (s)', color='royalblue')
ax2_twin = ax2.twinx()
rects2 = ax2_twin.bar(x_indices + width/2, front_sizes, width, label='Pareto Front Size', color='mediumseagreen')
ax2.set_ylabel('Runtime (seconds)', color='royalblue')
ax2_twin.set_ylabel('Number of Solutions', color='mediumseagreen')
ax2.set_xticks(x_indices)
ax2.set_xticklabels(algos)
ax2.set_title('Computational Efficiency & Solution Yield')
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=85)
if res_nsga2.F is not None:
    axes[0].scatter(res_nsga2.F[:, 0], res_nsga2.F[:, 1], c='crimson', label='NSGA-II', alpha=0.7)
    axes[1].scatter(res_nsga2.F[:, 0], res_nsga2.F[:, 2], c='crimson', label='NSGA-II', alpha=0.7)
    axes[2].scatter(res_nsga2.F[:, 1], res_nsga2.F[:, 2], c='crimson', label='NSGA-II', alpha=0.7)
if res_mopso.F is not None:
    axes[0].scatter(res_mopso.F[:, 0], res_mopso.F[:, 1], c='teal', label='MOPSO-CD', alpha=0.7)
    axes[1].scatter(res_mopso.F[:, 0], res_mopso.F[:, 2], c='teal', label='MOPSO-CD', alpha=0.7)
    axes[2].scatter(res_mopso.F[:, 1], res_mopso.F[:, 2], c='teal', label='MOPSO-CD', alpha=0.7)
axes[0].set_xlabel('Makespan ($f_1$)')
axes[0].set_ylabel('Workload Imbalance ($f_2$)')
axes[0].set_title('Makespan vs Workload Imbalance')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].set_xlabel('Makespan ($f_1$)')
axes[1].set_ylabel('Coordination Risk ($f_3$)')
axes[1].set_title('Makespan vs Coordination Risk')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

axes[2].set_xlabel('Workload Imbalance ($f_2$)')
axes[2].set_ylabel('Coordination Risk ($f_3$)')
axes[2].set_title('Workload Imbalance vs Coordination Risk')
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### 6. Complete 200-Task Project Schedules & Visual Gantt Charts (3 Objective-Based Strategies)
This section decodes the **complete 200-task project schedule** ($task_0 \dots task_{199}$) for 3 distinct objective-based decision strategies:

1. **Strategy A (Fastest Makespan):** $\min(f_1)$ — Priority on completing all 200 tasks in minimum duration.
2. **Strategy B (Fairest Workload Allocation):** $\min(f_2)$ — Priority on equalizing developer workloads.
3. **Strategy C (Lowest Coordination Breakdown Risk):** $\min(f_3)$ — Priority on minimizing cross-team communication breakdown risks.

In [ ]:
def decode_full_schedule(solution_x, problem):
    n_tasks = problem.n_tasks
    assigned_devs = []
    task_durations = np.zeros(n_tasks)
    
    for i in range(n_tasks):
        q_devs = problem.qualified_devs_per_task[i]
        q_idx = int(np.floor((solution_x[i] / problem.n_devs) * len(q_devs))) % len(q_devs)
        dev = q_devs[q_idx]
        assigned_devs.append(dev)
        task = problem.tasks[i]
        dur = problem.dev_time(task, dev)
        task_durations[i] = dur
        
    delays = solution_x[n_tasks:]
    start_times = np.zeros(n_tasks)
    finish_times = np.zeros(n_tasks)
    dev_free_time = np.zeros(problem.n_devs)
    pred_map = {i: [] for i in range(n_tasks)}
    for dep in problem.dependencies:
        if dep.dep_type != 'cyclic':
            pred_map[dep.succ_id].append(dep.pred_id)
        
    for i in range(n_tasks):
        assigned_dev = assigned_devs[i]
        dur = task_durations[i]
        dep_ready = max([finish_times[p] for p in pred_map[i]], default=0.0)
        dev_ready = dev_free_time[assigned_dev.dev_id]
        earliest_start = max(dep_ready, dev_ready)
        start_i = earliest_start + delays[i]
        finish_i = start_i + dur
        start_times[i] = start_i
        finish_times[i] = finish_i
        dev_free_time[assigned_dev.dev_id] = finish_i
        
    schedule_records = []
    for i in range(n_tasks):
        dev = assigned_devs[i]
        task = problem.tasks[i]
        schedule_records.append({
            'Task_ID': task.task_id,
            'Task_Name': task.name,
            'Req_Skill': task.req_skill,
            'Story_Points': task.story_points,
            'Assigned_Dev': dev.name,
            'Dev_Team': f'Team_{dev.team_id+1}',
            'Dev_Exp': dev.exp_level,
            'Start_Time': round(start_times[i], 2),
            'Finish_Time': round(finish_times[i], 2),
            'Duration': round(task_durations[i], 2)
        })
    return pd.DataFrame(schedule_records)

def plot_gantt_chart(df_sched, title='Project Execution Schedule (Gantt Chart)'):
    plt.figure(figsize=(14, 6), dpi=85)
    devs = df_sched['Assigned_Dev'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(devs)))
    dev_color_map = {dev: colors[i] for i, dev in enumerate(devs)}
    
    for _, row in df_sched.iterrows():
        dev = row['Assigned_Dev']
        color = dev_color_map[dev]
        start = row['Start_Time']
        dur = row['Duration']
        tid = row['Task_ID']
        
        plt.barh(dev, dur, left=start, height=0.55, align='center', color=color, edgecolor='black', alpha=0.85)
        plt.text(start + dur / 2.0, dev, f't{tid}', va='center', ha='center', fontsize=5.5, color='black', fontweight='bold')
        
    plt.xlabel('Project Timeline (Time Units / Hours)', fontsize=10, fontweight='bold')
    plt.ylabel('Assigned Developer', fontsize=10, fontweight='bold')
    plt.title(title, fontsize=11, fontweight='bold', pad=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

def get_strategic_indices(res):
    F = res.F
    if F is None or len(F) == 0:
        return None, None, None
    idx_a = np.argmin(F[:, 0]) # Strategy A: min f1 (Makespan)
    idx_b = np.argmin(F[:, 1]) # Strategy B: min f2 (Workload Imbalance)
    idx_c = np.argmin(F[:, 2]) # Strategy C: min f3 (Coordination Breakdown Risk)
    return idx_a, idx_b, idx_c

print('Schedule decoding helper functions defined.')


In [ ]:
# NSGA-II Strategic Schedules Output (Full 200 Tasks)
idx_a_n, idx_b_n, idx_c_n = get_strategic_indices(res_nsga2)

if idx_a_n is not None:
    print('========================================================================================')
    print('  NSGA-II STRATEGY A: FASTEST MAKESPAN SCHEDULE (min f1)  [f1=%.2f, f2=%.2f, f3=%.2f]' % tuple(res_nsga2.F[idx_a_n]))
    print('========================================================================================')
    df_sched_a = decode_full_schedule(res_nsga2.X[idx_a_n], problem)
    display(df_sched_a.head(15))
    plot_gantt_chart(df_sched_a, title=f'NSGA-II Strategy A (min f1 Makespan: {res_nsga2.F[idx_a_n][0]:.2f})')

    print('========================================================================================')
    print('  NSGA-II STRATEGY C: LOWEST RISK SCHEDULE (min f3)  [f1=%.2f, f2=%.2f, f3=%.2f]' % tuple(res_nsga2.F[idx_c_n]))
    print('========================================================================================')
    df_sched_c = decode_full_schedule(res_nsga2.X[idx_c_n], problem)
    display(df_sched_c.head(15))
    plot_gantt_chart(df_sched_c, title=f'NSGA-II Strategy C (min f3 Coordination Risk: {res_nsga2.F[idx_c_n][2]:.2f})')
